In [ ]:
# Installation des packages

import sys

!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install matplotlib tqdm seaborn scikit-learn

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
import pandas as pd
import numpy as np
from PIL import Image
import os
from tqdm import tqdm

In [3]:
# Dataset VinDr
class VinDrDataset(Dataset):
    def __init__(self, df, images_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, row["study_id"], row["image_id"] + ".png")
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, row["label"]

In [4]:
# Téléchargement depuis S3
!mc cp "s3/hlegarzic/stat_app/best_resnet18_model.pth" "/home/onyxia/best_resnet18_model.pth"
!mc cp "s3/hlegarzic/stat_app/breast-level_annotations.csv" "/home/onyxia/breast-level_annotations.csv"
!mc mirror "s3/hlegarzic/stat_app/images_png" "/home/onyxia/images_png"
!mc mirror "s3/hlegarzic/stat_app/MINI-DDSM-Complete-JPEG-82/" "/home/onyxia/data/ddsm/"

# Chargement des données
df = pd.read_csv("/home/onyxia/breast-level_annotations.csv")
df = df[df["view_position"] == "MLO"]
df = df[df["breast_birads"].isin(["BI-RADS 1", "BI-RADS 2", "BI-RADS 3", "BI-RADS 4", "BI-RADS 5"])]
df["label"] = (df["breast_birads"] == "BI-RADS 1").astype(int)  # 1=normal, 0=cancer

# Vérification que l'image existe
images_dir = "/home/onyxia/images_png"
df = df[df.apply(lambda r: os.path.exists(
    os.path.join(images_dir, r["study_id"], r["image_id"] + ".png")), axis=1)]

# Split train/val
from sklearn.model_selection import train_test_split
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

transform = transforms.Compose([
    transforms.Resize((700, 700)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = VinDrDataset(df_train, images_dir, transform)
val_dataset   = VinDrDataset(df_val,   images_dir, transform)
train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=2)
val_loader    = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=2)

 0 B / ? ┃░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░▓┃ 0s]11;?\]11;?\]11;?\

In [ ]:
# Chargement du modèle
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation de : {device}")  # pour vérifier l'utilisation du GPU
model = models.resnet18()
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load("/home/onyxia/best_resnet18_model.pth", map_location=device))
model = model.to(device)

Utilisation de : cuda


In [ ]:
# Calcul de la Fisher Information Matrix (sur DDSM)
def compute_fisher(model, data_loader, device, n_batches=20):  # 50 → 20
    fisher = {n: torch.zeros_like(p) for n, p in model.named_parameters() if p.requires_grad}
    model.eval()
    for i, (inputs, labels) in enumerate(tqdm(data_loader, desc="Calcul Fisher")):
        if i >= n_batches:
            break
        inputs, labels = inputs.to(device), labels.to(device)
        model.zero_grad()
        outputs = model(inputs)
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        for n, p in model.named_parameters():
            if p.requires_grad and p.grad is not None:
                fisher[n] += p.grad.data ** 2
    for n in fisher:
        fisher[n] /= min(n_batches, len(data_loader))
    return fisher

ddsm_dataset = ImageFolder("/home/onyxia/data/ddsm/", transform=transform)
ddsm_loader = DataLoader(ddsm_dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True,
    persistent_workers=True)

fisher = compute_fisher(model, ddsm_loader, device)

# Sauvegarde des poids originaux (θ*)
params_star = {n: p.clone().detach() for n, p in model.named_parameters() if p.requires_grad}

Calcul Fisher:   0%|          | 0/58 [00:00<?, ?it/s]

Calcul Fisher:  34%|███▍      | 20/58 [00:47<01:29,  2.36s/it]


In [7]:
# Finetuning EWC
def ewc_loss_fast(model, fisher, fisher_times_star, fisher_times_star_sq, lambda_ewc=1000):
    loss = 0
    for n, p in model.named_parameters():
        if n in fisher:
            loss += (fisher[n] * p ** 2 - 2 * fisher_times_star[n] * p).sum()
    return lambda_ewc * loss

lambda_ewc = 1000
num_epochs = 10
best_val_acc = 0.0

fisher_times_star = {n: fisher[n] * params_star[n] for n in fisher}
fisher_times_star_sq = {n: fisher[n] * params_star[n] ** 2 for n in fisher}
constant_term = lambda_ewc * sum((fisher_times_star_sq[n]).sum() for n in fisher_times_star_sq)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(inputs)
        loss = F.cross_entropy(outputs, labels) \
             + ewc_loss_fast(model, fisher, fisher_times_star, fisher_times_star_sq) \
             + constant_term
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, preds = torch.max(model(inputs), 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total * 100
    print(f"Epoch {epoch+1} | Loss: {train_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/home/onyxia/best_resnet18_ewc.pth")
        print(f"  → Meilleur modèle sauvegardé ({val_acc:.2f}%)")

print(f"\nMeilleure Val Acc : {best_val_acc:.2f}%")

# Sauvegarde
!mc cp "/home/onyxia/best_resnet18_ewc.pth" "s3/hlegarzic/stat_app/best_resnet18_ewc.pth"

Epoch 1/10: 100%|██████████| 500/500 [05:06<00:00,  1.63it/s]


Epoch 1 | Loss: 0.6579 | Val Acc: 77.00%
  → Meilleur modèle sauvegardé (77.00%)


Epoch 2/10: 100%|██████████| 500/500 [05:05<00:00,  1.63it/s]


Epoch 2 | Loss: 0.5660 | Val Acc: 77.75%
  → Meilleur modèle sauvegardé (77.75%)


Epoch 3/10: 100%|██████████| 500/500 [05:04<00:00,  1.64it/s]


...

Epoch 8 | Loss: 0.5219 | Val Acc: 76.65%


Epoch 9/10: 100%|██████████| 500/500 [05:06<00:00,  1.63it/s]


Epoch 9 | Loss: 0.4655 | Val Acc: 68.45%


Epoch 10/10: 100%|██████████| 500/500 [05:06<00:00,  1.63it/s]


Epoch 10 | Loss: 0.4639 | Val Acc: 76.90%

Meilleure Val Acc : 78.85%
...18_ewc.pth: 42.72 MiB / 42.72 MiB ┃▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓┃ 129.82 MiB/s 0s